# PLANTILLA SET UP

**IMPORTANTE**: Recuerda hacer una copia de esta plantilla para no machacar la original.

## CREAR EL ENTORNO PARA EL PROYECTO

Crear un nuevo entorno e instalar los paquetes básicos:

**1. Ir a la carpeta del proyecto mediante el prompt o terminal**
```cmd
uv venv --python 3.11 .venv 
```

**2. Copiar y poner la línea inferior:**
```cmd
uv pip install \
  numpy \
  pandas==1.5.3 \
  pandas_flavor==0.1.2 \
  matplotlib \
  seaborn \
  scikit-learn==1.3.1 \
  scipy==1.9.3 \
  sqlalchemy \
  xgboost==2.0.3 \
  pyjanitor==0.22.0 \
  scikit-plot \
  yellowbrick \
  imbalanced-learn \
  cloudpickle \
  notebook==5.7.8 \
  jupyter_client \
  jupyter_contrib_nbextensions \
  category_encoders==2.6.2 \
  plotly
```

**3. Crear el environment.yml / requirements.txt**
```cmd
uv pip freeze > requirements.txt
conda env export > environment.yml
``` 

## IMPORTAR PAQUETES

In [188]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

# Autocomplete settings
%config IPCompleter.greedy=True

## CREAR EL DIRECTORIO DEL PROYECTO

In [189]:
root_path = Path('/Users/gloriadelriomarquez/Documents/Career/Projects/ml-lab/')
project = 'client_churn'
path = root_path / project

In [190]:
# Set working directory to project
os.chdir(path)

## IMPORT DATA

¿Qué haría?
1. Descargar transactions.csv y guardarlo en 02_data/raw/.
2. Comprobar el rango de fechas y el número de operaciones por mes de ambos archivos.
3. Revisar si existen transacciones repetidas entre los dos.
4. Unirlos con pd.concat() después de comprobar que tienen las mismas columnas y gestionar los posibles duplicados.
5. Filtrar por la fecha de corte y construir las features sobre el histórico combinado.
No descargaría todavía user_logs.csv: añade bastante volumen y complejidad.
Eso sí, hay una decisión que tomar: si quieres terminar un primer modelo rápido, puedes empezar solo con train y members, sin transacciones. Pero si quieres trabajar las variables transaccionales que acabamos de diseñar, sí incorporaría transactions.csv ahora, antes de seguir con el feature engineering.
    

Poner los datos originales en la carpeta '/02_data/raw'

### Importar los datos

In [191]:
# Localize data files
data_path = Path(path) / "02_data/raw"

data_files = []
data_files = [file_path.name for file_path in sorted(data_path.glob("*.csv"))]

data_files

['members_v3.csv',
 'sample_submission_zero.csv',
 'train.csv',
 'train_v2.csv',
 'transactions.csv',
 'transactions_v2.csv']

In [192]:
# Import files
dfs = {}

for file in data_files:
    name = Path(file).stem
    dfs[name] = pd.read_csv(data_path / file)

    print(f"{name}: {dfs[name].shape}")
    display(dfs[name].head())

members_v3: (6769473, 6)


,msno,city,bd,gender,registered_via,registration_init_time
0,Rb9UwLQTrxzBVwCB6+bCcSQWZ9JiNLC9dXtM1oEsZA8=,1,0,NaN,11,20110911
1,+tJonkh+O1CA796Fm5X60UMOtB6POHAwPjbTRVl/EuU=,1,0,NaN,7,20110914
2,cV358ssn7a0f7jZOwGNWS07wCKVqxyiImJUX6xcIwKw=,1,0,NaN,11,20110915
3,9bzDeJP6sQodK73K5CBlJ6fgIQzPeLnRl0p5B77XP+g=,1,0,NaN,11,20110915
4,WFLY3s7z4EZsieHCt63XrsdtfTEmJ+2PnnKLH5GY4Tk=,6,32,female,9,20110915


sample_submission_zero: (970960, 2)


,msno,is_churn
0,ugx0CjOMzazClkFzU2xasmDZaoIqOUAZPsH1q0teWCg=,0
1,f/NmvEzHfhINFEYZTR05prUdr+E+3+oewvweYz9cCQE=,0
2,zLo9f73nGGT1p21ltZC3ChiRnAVvgibMyazbCxvWPcg=,0
3,8iF/+8HY8lJKFrTc7iR9ZYGCG2Ecrogbc2Vy5YhsfhQ=,0
4,K6fja4+jmoZ5xG6BypqX80Uw/XKpMgrEMdG2edFOxnA=,0


train: (992931, 2)


,msno,is_churn
0,waLDQMmcOu2jLDaV1ddDkgCrB/jl6sD66Xzs0Vqax1Y=,1
1,QA7uiXy8vIbUSPOkCf9RwQ3FsT8jVq2OxDr8zqa7bRQ=,1
2,fGwBva6hikQmTJzrbz/2Ezjm5Cth5jZUNvXigKK2AFA=,1
3,mT5V8rEpa+8wuqi6x0DoVd3H5icMKkE9Prt49UlmK+4=,1
4,XaPhtGLk/5UvvOYHcONTwsnH97P4eGECeq+BARGItRw=,1


... (2 outputs left out)

transactions: (21547746, 9)


,msno,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,transaction_date,membership_expire_date,is_cancel
0,YyO+tlZtAXYXoZhNr3Vg3+dfVQvrBVGO8j1mfqe4ZHc=,41,30,129,129,1,20150930,20151101,0
1,AZtu6Wl0gPojrEQYB8Q3vBSmE2wnZ3hi1FbK1rQQ0A4=,41,30,149,149,1,20150930,20151031,0
2,UkDFI97Qb6+s2LWcijVVv4rMAsORbVDT2wNXF0aVbns=,41,30,129,129,1,20150930,20160427,0
3,M1C56ijxozNaGD0t2h68PnH2xtx5iO5iR2MVYQB6nBI=,39,30,149,149,1,20150930,20151128,0
4,yvj6zyBUaqdbUQSrKsrZ+xNDVM62knauSZJzakS9OW4=,39,30,149,149,1,20150930,20151121,0


transactions_v2: (1431009, 9)


,msno,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,transaction_date,membership_expire_date,is_cancel
0,++6eU4LsQ3UQ20ILS7d99XK8WbiVgbyYL4FUgzZR134=,32,90,298,298,0,20170131,20170504,0
1,++lvGPJOinuin/8esghpnqdljm6NXS8m8Zwchc7gOeA=,41,30,149,149,1,20150809,20190412,0
2,+/GXNtXWQVfKrEDqYAzcSw2xSPYMKWNj22m+5XkVQZc=,36,30,180,180,1,20170303,20170422,0
3,+/w1UrZwyka4C9oNH3+Q8fUf3fD8R3EwWrx57ODIsqk=,36,30,180,180,1,20170329,20170331,1
4,+00PGzKTYqtnb65mPKPyeHXcZEwqiEzktpQksaaSC3c=,41,30,99,99,1,20170323,20170423,0


<div style="background-color: #162B36; color: #E6EDF3; padding: 20px; border-radius: 12px; border-left: 5px solid #38BDF8;">

### Descripción de las variables de `transactions`

- **`msno`**: identificador único del cliente. Un mismo cliente puede tener múltiples transacciones a lo largo del tiempo.
- **`payment_method_id`**: identificador numérico del método de pago utilizado (tarjeta, transferencia, etc.). Es una variable categórica; el código no indica necesariamente un orden.
- **`payment_plan_days`**: duración nominal del plan de suscripción contratado, expresada en días. Por ejemplo, 30 días para un plan mensual o 365 días para uno anual.
- **`plan_list_price`**: precio oficial o de catálogo del plan contratado, antes de posibles descuentos o promociones.
- **`actual_amount_paid`**: importe realmente pagado por el cliente en esa transacción. Puede ser inferior a `plan_list_price` si existe algún descuento.
- **`is_auto_renew`**: indicador binario que señala si la renovación automática está activada en la transacción:
  - `1`: renovación automática activada.
  - `0`: renovación automática desactivada.
- **`transaction_date`**: fecha en la que se registra la transacción. Puede corresponder a una contratación, renovación, cancelación o modificación de la suscripción.
- **`membership_expire_date`**: fecha de vencimiento de la suscripción registrada en esa transacción. Puede ser posterior a `transaction_date` y representar una fecha futura conocida en el momento de la operación.
- **`is_cancel`**: indicador binario que señala si la transacción corresponde a una cancelación:
  - `1`: operación de cancelación.
  - `0`: operación no marcada como cancelación.

**Nota:** `is_cancel` no equivale necesariamente a `is_churn`. Un cliente puede cancelar una operación y posteriormente renovar o contratar otra suscripción. `is_churn` es la variable objetivo que indica el abandono del cliente según la definición de la competición.

</div>

### Inventario y visión global de datasets

In [193]:
dfs.keys()

dict_keys(['members_v3', 'sample_submission_zero', 'train', 'train_v2', 'transactions', 'transactions_v2'])

In [194]:
train = dfs["train"]
train_v2 = dfs["train_v2"]
members = dfs["members_v3"]
transactions = dfs["transactions"]
transactions_v2 = dfs["transactions_v2"]

In [195]:
for nombre, df in dfs.items():
    print(f"\n{nombre.upper()}")
    print(f"Filas: {len(df):,}")
    print(f"Clientes únicos: {df['msno'].nunique():,}")
    print(f"Duplicados de msno: {df['msno'].duplicated().sum():,}")

clientes_train = set(train["msno"])

print("\n")

for nombre, df in [("members", members), ("transactions", transactions)]:
    cobertura = train["msno"].isin(df["msno"]).mean()
    print(f"Cobertura de {nombre} sobre train: {cobertura:.2%}")

for nombre, df in [("members", members), ("transactions_v2", transactions_v2)]:
    cobertura_v2 = train_v2["msno"].isin(df["msno"]).mean()
    print(f"Cobertura de {nombre} sobre train_v2: {cobertura_v2:.2%}")

for name in ["transactions", "transactions_v2"]:
    df = dfs[name]
    print(f"\nRango de fechas de {name}")
    fechas = pd.to_datetime(df["transaction_date"].astype(str), format="%Y%m%d")
    print(fechas.min(), fechas.max())



MEMBERS_V3
Filas: 6,769,473
Clientes únicos: 6,769,473
Duplicados de msno: 0

SAMPLE_SUBMISSION_ZERO
Filas: 970,960
Clientes únicos: 970,960
Duplicados de msno: 0

TRAIN
Filas: 992,931
Clientes únicos: 992,931
Duplicados de msno: 0

TRAIN_V2
Filas: 970,960
Clientes únicos: 970,960
Duplicados de msno: 0

TRANSACTIONS
Filas: 21,547,746
Clientes únicos: 2,363,626
Duplicados de msno: 19,184,120

TRANSACTIONS_V2
Filas: 1,431,009
Clientes únicos: 1,197,050
Duplicados de msno: 233,959


Cobertura de members sobre train: 88.34%
Cobertura de transactions sobre train: 100.00%
Cobertura de members sobre train_v2: 88.67%
Cobertura de transactions_v2 sobre train_v2: 96.15%

Rango de fechas de transactions
2015-01-01 00:00:00 2017-02-28 00:00:00

Rango de fechas de transactions_v2
2015-01-01 00:00:00 2017-03-31 00:00:00


<div style="background-color: #162B36; color: #E6EDF3; padding: 20px; border-radius: 12px; border-left: 5px solid #38BDF8;">

## EDA

### Análisis de las tablas

| Tabla | Filas | Clientes únicos | Duplicados por `msno` |
|---|---:|---:|---:|
| `members_v3` | 6.769.473 | 6.769.473 | 0 |
| `train` | 992.931 | 992.931 | 0 |
| `train_v2` | 970.960 | 970.960 | 0 |
| `transactions` | 21.547.746 | 2.363.626 | 19.184.120 |
| `transactions_v2` | 1.431.009 | 1.197.050 | 233.959 |

**Interpretación:**

- `train` y `train_v2` contienen una fila por cliente y la variable objetivo `is_churn`.
- `members_v3` contiene información demográfica sin duplicados por cliente, por lo que permite realizar un `LEFT JOIN`.
- `transactions` y `transactions_v2` contienen múltiples operaciones por cliente, por lo que será necesario agregarlas por `msno` antes de unirlas.
- Los duplicados por `msno` representan filas adicionales por cliente, no necesariamente registros erróneos.

**Cobertura de datos:**

| Tabla principal | Tabla relacionada | Cobertura |
|---|---|---:|
| `train` | `members_v3` | 88,34 % |
| `train` | `transactions` | 100,00 % |
| `train_v2` | `members_v3` | 88,67 % |
| `train_v2` | `transactions_v2` | 96,15 % |

Las coberturas se han calculado antes de aplicar filtros temporales.

---

### Periodos de predicción y etiquetas

- **`train.csv`:** contiene las etiquetas reales de churn correspondientes a febrero de 2017.
- **`train_v2.csv`:** contiene las etiquetas reales de churn correspondientes a marzo de 2017.

**Decisión:** utilizaremos inicialmente `train_v2.csv` para desarrollar un modelo que prediga el churn de marzo de 2017.

La variable objetivo es `is_churn`:

- `1`: el cliente abandona el servicio.
- `0`: el cliente permanece.

Como disponemos de las etiquetas reales, podremos dividir los datos en entrenamiento, validación y test y evaluar el rendimiento del modelo mediante métricas como precision, recall, F1-score y ROC-AUC.

**El objetivo es simular que nos encontramos a finales de febrero de 2017 y queremos anticipar qué clientes abandonarán en marzo.**

---

### Análisis temporal y prevención de data leakage

| Archivo | Primera transacción | Última transacción |
|---|---|---|
| `transactions.csv` | 01/01/2015 | 28/02/2017 |
| `transactions_v2.csv` | 01/01/2015 | 31/03/2017 |

Para evitar *data leakage*, se establece el **28/02/2017 como último día de información observable**.

- Se utilizarán únicamente transacciones con `transaction_date` anterior al 01/03/2017.
- Se excluirán las operaciones realizadas durante marzo.
- Las fechas futuras de vencimiento podrán utilizarse si ya eran conocidas antes del punto de corte.
- Antes de combinar ambos históricos, se comprobarán los posibles registros duplicados y la compatibilidad de sus columnas.

---

### Estrategia de integración de datos

1. Utilizar `train_v2` como tabla principal, conservando una fila por cliente.
2. Incorporar `members_v3` mediante un `LEFT JOIN` por `msno`.
3. Combinar `transactions` y `transactions_v2`, revisando previamente los solapamientos.
4. Filtrar las transacciones según la fecha de corte.
5. Construir features transaccionales por cliente: frecuencia, recencia, importes, renovaciones, cancelaciones y vencimientos.
6. Incorporar las features agregadas mediante un `LEFT JOIN`.
7. Comprobar la cardinalidad, los valores nulos y la calidad del dataset resultante.

**Objetivo final:** obtener una tabla analítica con una fila por cliente, sus variables predictoras y la etiqueta `is_churn`, preparada para el EDA, el preprocesamiento y el entrenamiento de modelos de Machine Learning.

Como posible ampliación, se podrá utilizar `train.csv` para entrenar con datos de febrero y evaluar sobre marzo (`train_v2`), construyendo las features con el corte temporal correspondiente a cada periodo.

</div>

In [196]:
nulls = pd.DataFrame({
    "nulls_abs": members.isna().sum(),
    "nulls_pct": members.isna().mean() 
}).sort_values("nulls_pct", ascending=False)

nulls

,nulls_abs,nulls_pct
gender,4429505,0.654335
msno,0,0.000000
city,0,0.000000
bd,0,0.000000
registered_via,0,0.000000
registration_init_time,0,0.000000


In [197]:
transactions_v2.columns

Index(['msno', 'payment_method_id', 'payment_plan_days', 'plan_list_price',
       'actual_amount_paid', 'is_auto_renew', 'transaction_date',
       'membership_expire_date', 'is_cancel'],
      dtype='str')

In [198]:
# Convertimos a formato datetime las columnas de fechas de transaction 
date_cols = transactions.columns[transactions.columns.str.contains('date')]

for name in ["transactions", "transactions_v2"]:
    df = dfs[name]

    df[date_cols] = df[date_cols].apply(
        lambda c: pd.to_datetime(c.astype("Int64").astype(str), format="%Y%m%d", errors= "raise")
    )

In [199]:
# Filtramos las transacciones según la fecha de corte (evitar data leakage)
transactions_filtered = transactions[transactions["transaction_date"] < "2017-03-01"].copy()
transactions_v2_filtered = transactions_v2[transactions_v2["transaction_date"] < "2017-03-01"].copy()

# Comprobamos
max_date = transactions_filtered["transaction_date"].max()
max_date_v2 = transactions_v2_filtered["transaction_date"].max()

n_descartadas = len(transactions) - len(transactions_filtered)
n_descartadas_v2 = len(transactions_v2) - len(transactions_v2_filtered)

print(f"Transacción más reciente en v1: {max_date:%d/%m/%Y}")  # debería ser <= 28/02/2017
print(f"Filas descartadas en v1: {n_descartadas:,}\n".replace(",", "."))

print(f"Transacción más reciente en v2: {max_date_v2:%d/%m/%Y}")  # debería ser <= 28/02/2017
print(f"Filas descartadas en v2: {n_descartadas_v2:,}\n".replace(",", "."))


for nombre, df in [
    ("transactions original", transactions),
    ("transactions_v2 original", transactions_v2),
    ("transactions filtrado", transactions_filtered),
    ("transactions_v2 filtrado", transactions_v2_filtered),
]:
    print(f"{nombre}:")
    print(f"  Transacciones: {len(df):,}")
    print(f"  Clientes únicos: {df['msno'].nunique():,}")
    print("")
    
# Clientes comunes antes y después del filtro
comunes_original = len(
    set(transactions["msno"]) & set(transactions_v2["msno"])
)

comunes_filtrado = len(
    set(transactions_filtered["msno"]) &
    set(transactions_v2_filtered["msno"])
)

print(f"\nClientes comunes originales: {comunes_original:,}")
print(f"Clientes comunes tras filtrar: {comunes_filtrado:,}")

Transacción más reciente en v1: 28/02/2017
Filas descartadas en v1: 0

Transacción más reciente en v2: 28/02/2017
Filas descartadas en v2: 1.069.822

transactions original:
  Transacciones: 21,547,746
  Clientes únicos: 2,363,626

transactions_v2 original:
  Transacciones: 1,431,009
  Clientes únicos: 1,197,050

transactions filtrado:
  Transacciones: 21,547,746
  Clientes únicos: 2,363,626

transactions_v2 filtrado:
  Transacciones: 361,187
  Clientes únicos: 262,429


Clientes comunes originales: 1,134,533
Clientes comunes tras filtrar: 234,380


In [200]:
# Combinamos transactions y transactions_v2, revisando previamente los solapamientos
tx1 = transactions_filtered
tx2 = transactions_v2_filtered

# 1. Comprobar compatibilidad de columnas
print("Mismas columnas:", set(tx1.columns) == set(tx2.columns))

# 2. Clientes presentes en ambos archivos
clientes_comunes = pd.Index(tx1["msno"].unique()).intersection(
    pd.Index(tx2["msno"].unique())
)

print(f"Clientes comunes: {len(clientes_comunes):,}\n")

# 3. Duplicados exactos dentro de cada archivo
print(f"Duplicados internos transactions: {tx1.duplicated().sum():,}")
print(f"Duplicados internos transactions_v2: {tx2.duplicated().sum():,}\n")

# 4. Coincidencias exactas entre archivos
columnas = sorted(tx1.columns)
hash_tx1 = pd.util.hash_pandas_object(tx1[columnas], index=False)
hash_tx2 = pd.util.hash_pandas_object(tx2[columnas], index=False)

coincidencias = hash_tx2.isin(pd.Index(hash_tx1.unique()))
print(f"Filas de v2 presentes en transactions: {coincidencias.sum():,}")

Mismas columnas: True
Clientes comunes: 234,380

Duplicados internos transactions: 3,339
Duplicados internos transactions_v2: 0

Filas de v2 presentes en transactions: 0


In [201]:
# Eliminamos dupliados 
transactions_filtered = transactions_filtered.drop_duplicates()

# Concatenamos los históricos filtrados
transactions_all = pd.concat(
    [transactions_filtered, transactions_v2_filtered],
    ignore_index=True
)

# Comprobar dimensiones
print(f"Transacciones: {len(transactions_all):,}")
print(f"Clientes únicos: {transactions_all['msno'].nunique():,}")

# Comprobar cobertura sobre train_v2
cobertura = train_v2["msno"].isin(transactions_all["msno"]).mean()
print(f"Cobertura sobre train_v2: {cobertura:.2%}")


Transacciones: 21,905,594
Clientes únicos: 2,391,675
Cobertura sobre train_v2: 99.74%


### Actividad transanccional 

<div style="background-color: #162B36; color: #E6EDF3; padding: 20px; border-radius: 12px; border-left: 5px solid #38BDF8;">

Comprobamos si existen múltiples transacciones dentro de un mismo día para un mismo cliente

</div>

In [202]:
# Número de transacciones por cliente y fecha
tx_same_day = (
    transactions_all
    .groupby(["msno", "transaction_date"])
    .size()
    .reset_index(name="n_transactions")
)

# Seleccionar combinaciones cliente-fecha con más de una transacción
tx_same_day = tx_same_day.query("n_transactions > 1")

# Métricas
n_clients = tx_same_day["msno"].nunique()
n_days = len(tx_same_day)
n_transactions = tx_same_day["n_transactions"].sum()

print(f"Clientes con varias transacciones el mismo día: {n_clients:,}")
print(f"Porcentaje de clientes: {n_clients / transactions_all['msno'].nunique():.2%}\n")

print(f"Transacciones implicadas: {n_transactions:,}")
print(f"Porcentaje de transacciones: {n_transactions / len(transactions_all):.2%}")

# Distribución del número de operaciones en un mismo día
display(pd.DataFrame(
    tx_same_day["n_transactions"]
    .value_counts()
    .sort_index()
    .rename_axis("transacciones_por_dia")
    .reset_index(name="n_casos")
    ).T
)

# Ejemplos de clientes con varias operaciones el mismo día
display(
    transactions_all.merge(
        tx_same_day[["msno", "transaction_date"]],
        on=["msno", "transaction_date"],
        how="inner"
    )
    .sort_values(["msno", "transaction_date"])
    .head(8)
)

Clientes con varias transacciones el mismo día: 218,684
Porcentaje de clientes: 9.14%

Transacciones implicadas: 591,912
Porcentaje de transacciones: 2.70%


,0,1,2,3,4,5,6,7,8,9,...,53,54,55,56,57,58,59,60,61,62
transacciones_por_dia,2,3,4,5,6,7,8,9,10,11,...,74,76,82,89,100,118,124,134,145,156
n_casos,258351,16880,2681,457,298,116,85,67,54,29,...,1,1,1,1,1,1,1,1,1,1


,msno,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,transaction_date,membership_expire_date,is_cancel
135389,++0BJXY8tpirgIhJR14LDM1pnaRosjD1mdO1mIKxlJA=,38,30,149,149,0,2015-09-02,2015-10-02,0
249301,++0BJXY8tpirgIhJR14LDM1pnaRosjD1mdO1mIKxlJA=,35,7,0,0,0,2015-09-02,2015-10-12,0
185701,++2axpngZEynlxNr1+AkwgHHfaEZ/EeOj6Q284RiAkw=,41,30,149,149,1,2016-02-25,2016-02-25,1
532892,++2axpngZEynlxNr1+AkwgHHfaEZ/EeOj6Q284RiAkw=,35,7,0,0,0,2016-02-25,2016-03-25,0
276766,++3Z+W8OPnpbHYfrKwqRKN1bF83XEbxjdYUolhGdHZg=,41,0,0,129,1,2015-05-04,2015-06-09,0
537072,++3Z+W8OPnpbHYfrKwqRKN1bF83XEbxjdYUolhGdHZg=,41,0,0,149,1,2015-05-04,2015-06-04,1
27027,++4cUL0b9CfW8cj0A/wfSxQc4k4fcVtWcLqk2UOdpKs=,33,30,149,149,1,2016-10-28,2016-10-27,1
533044,++4cUL0b9CfW8cj0A/wfSxQc4k4fcVtWcLqk2UOdpKs=,26,1,0,0,0,2016-10-28,2016-11-28,0


<div style="background-color: #162B36; color: #E6EDF3; padding: 20px; border-radius: 12px; border-left: 5px solid #38BDF8;">
    
#### Tipos de actividad transaccional

Distinguimos entre dos tipos de actividad al analizar las transacciones de KKBox:

- **Actividad transaccional:** incluye todos los registros de operaciones, independientemente de si han sido canceladas. Permite analizar el número total de transacciones, las cancelaciones y la frecuencia de operaciones por día.

- **Actividad no cancelada:** considera únicamente las operaciones con `is_cancel = 0`. Permite calcular el número de transacciones no canceladas y los importes asociados a ellas.

**Nota:** una transacción cancelada no implica necesariamente un reembolso. Por tanto, la suma de `actual_amount_paid` no debe interpretarse directamente como el gasto neto real del cliente.

</div>

In [203]:
# Filtrar operaciones no canceladas
tx_active = transactions_all.loc[
    transactions_all["is_cancel"] == 0
].copy()

# Contar operaciones por cliente y fecha
same_day_active = (
    tx_active
    .groupby(["msno", "transaction_date"])
    .size()
)

# Casos con varias operaciones no canceladas el mismo día
same_day_active = same_day_active[same_day_active > 1]

print(f"Transacciones no canceladas: {len(tx_active):,}")
print(f"Clientes con varias operaciones no canceladas el mismo día: {same_day_active.index.get_level_values('msno').nunique():,}")
print(f"Porcentaje de clientes: { same_day_active.index.get_level_values('msno').nunique()/transactions_all['msno'].nunique():.2%}\n")

print(f"Transacciones implicadas: {same_day_active.sum():,}")
print(f"Porcentaje de transacciones: {same_day_active.sum()/len(tx_active):.2%}")

# Distribución de operaciones por cliente y día
display(pd.DataFrame(
    same_day_active.value_counts()
    .sort_index()
    .rename_axis("transacciones_por_dia")
    .reset_index(name="n_casos")
    ).T
)


Transacciones no canceladas: 21,045,855
Clientes con varias operaciones no canceladas el mismo día: 67,437
Porcentaje de clientes: 2.82%

Transacciones implicadas: 241,321
Porcentaje de transacciones: 1.15%


,0,1,2,3,4,5,6,7,8,9,...,53,54,55,56,57,58,59,60,61,62
transacciones_por_dia,2,3,4,5,6,7,8,9,10,11,...,74,76,82,89,100,118,124,134,145,156
n_casos,101163,6131,1764,418,291,111,84,66,53,29,...,1,1,1,1,1,1,1,1,1,1


<div style="background-color: #162B36; color: #E6EDF3; padding: 20px; border-radius: 12px; border-left: 5px solid #38BDF8;">

Muchas operaciones dentro del mismo día están relacionadas con cancelaciones, pero no todas.

En la distribución de arriba aparecen casos de hasta 156 operaciones no canceladas en un mismo día. Eso significa que las cancelaciones no explican por sí solas todos los registros múltiples. **Podrían existir otros mecanismos de facturación, operaciones repetidas o anomalías de calidad de datos.**

Antes de decidir cómo agregarlas, analizamos esos casos extremos.

</div>


In [204]:

# Analizar clientes y fechas con varias operaciones no canceladas
same_day_analysis = (
    tx_active
    .groupby(["msno", "transaction_date"])
    .agg(
        n_transactions=("msno", "size"),
        n_payment_methods=("payment_method_id", "nunique"),
        n_plans=("payment_plan_days", "nunique"),
        n_prices=("actual_amount_paid", "nunique"),
        n_expire_dates=("membership_expire_date", "nunique")
    )
    .query("n_transactions > 1")
)

# Cuántos grupos tienen diferencias en sus atributos
print(f"Grupos con distintos métodos de pago: {(same_day_analysis['n_payment_methods'] > 1).sum():,}")
print(f"Grupos con distintas duraciones de plan: {(same_day_analysis['n_plans'] > 1).sum():,}")
print(f"Grupos con distintos importes: {(same_day_analysis['n_prices'] > 1).sum():,}")
print(f"Grupos con distintas fechas de vencimiento: {(same_day_analysis['n_expire_dates'] > 1).sum():,}")

# Casos con mayor número de operaciones
display(
    same_day_analysis
    .sort_values("n_transactions", ascending=False)
    .head(10)
)


Grupos con distintos métodos de pago: 21,517
Grupos con distintas duraciones de plan: 20,824
Grupos con distintos importes: 46,370
Grupos con distintas fechas de vencimiento: 101,725


,,n_transactions,n_payment_methods,n_plans,n_prices,n_expire_dates
msno,transaction_date,,,,,
vf6eQrnFfiS9o1kB/gRUJ4iTUixS9tHNKizqQ/1vLDI=,2016-02-29,156,1,1,1,156
SNlFRAsmUqnXKPofSXA8WYUc5DtmLcUMy4pXSJ3Ohz0=,2016-02-29,145,1,1,1,145
72gJqt1O31E/WoxAEYFn9LHNI6mAZFGera5Q6gvsFkA=,2016-07-20,134,1,1,1,134
hQirtFbRr8t5T+/aVbrXM2tRNvoOIzChx5qzm1v/PE0=,2016-08-07,124,1,1,1,124
OGKDrZQDB3yewZhoSd5qqvmG5A1GcNTYMexO95NlH+g=,2016-10-31,118,1,1,1,118
5ty4nZkq54z93wQtBN7RHVYj8rNghBDCVBH+3xmxf0I=,2015-12-15,100,1,1,1,100
oyL1Yv3oa+GljYSpIVN1SqSY5xUmFxvyG2FutjyDQWw=,2016-08-13,89,1,1,1,89
cmlCHjCLeo0TQK1S/37OqQK1AvwJWkD2h5HBd9lATgw=,2016-07-20,82,1,1,1,82
5ty4nZkq54z93wQtBN7RHVYj8rNghBDCVBH+3xmxf0I=,2016-01-04,76,1,1,1,76


<div style="background-color: #162B36; color: #E6EDF3; padding: 20px; border-radius: 12px; border-left: 5px solid #38BDF8;">
    
En el primer cliente aparecen:
- 156 transacciones el mismo día.
- 1 único método de pago.
- 1 única duración de plan.
- 1 único importe.
- 156 fechas de vencimiento diferentes.
Esto es bastante peculiar. Parece que se están registrando múltiples periodos de suscripción en una misma fecha, aunque desconocemos la lógica exacta del sistema.
Podría tratarse de registros históricos, ajustes administrativos o alguna modalidad especial de facturación.
Lo que sí sabemos es que contar esas 156 filas como 156 compras independientes puede distorsionar nuestras features.

¿Las múltiples transacciones del mismo día representan periodos de suscripción diferentes o simplemente registros administrativos?
Para ello inspeccionamos uno de los casos extremos, especialmente la evolución de membership_expire_date.

</div>

In [205]:

cliente = "vf6eQrnFfiS9o1kB/gRUJ4iTUixS9tHNKizqQ/1vLDI="

tx_cliente = (
    transactions_all.loc[
        (transactions_all["msno"] == cliente) &
        (transactions_all["transaction_date"] == "2016-02-29")
    ]
    .sort_values("membership_expire_date")
)

display(tx_cliente.head(10))
display(tx_cliente.tail(10))

# Diferencia en días entre fechas de vencimiento consecutivas
display(
    tx_cliente["membership_expire_date"]
    .diff()
    .dt.days
    .value_counts()
    .sort_index()
)


,msno,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,transaction_date,membership_expire_date,is_cancel
12886785,vf6eQrnFfiS9o1kB/gRUJ4iTUixS9tHNKizqQ/1vLDI=,38,10,0,0,0,2016-02-29,2016-03-10,0
14891799,vf6eQrnFfiS9o1kB/gRUJ4iTUixS9tHNKizqQ/1vLDI=,38,10,0,0,0,2016-02-29,2016-03-20,0
19037291,vf6eQrnFfiS9o1kB/gRUJ4iTUixS9tHNKizqQ/1vLDI=,38,10,0,0,0,2016-02-29,2016-03-30,0
11833534,vf6eQrnFfiS9o1kB/gRUJ4iTUixS9tHNKizqQ/1vLDI=,38,10,0,0,0,2016-02-29,2016-04-09,0
11837105,vf6eQrnFfiS9o1kB/gRUJ4iTUixS9tHNKizqQ/1vLDI=,38,10,0,0,0,2016-02-29,2016-04-19,0
735892,vf6eQrnFfiS9o1kB/gRUJ4iTUixS9tHNKizqQ/1vLDI=,38,10,0,0,0,2016-02-29,2016-04-29,0
9387838,vf6eQrnFfiS9o1kB/gRUJ4iTUixS9tHNKizqQ/1vLDI=,38,10,0,0,0,2016-02-29,2016-05-09,0
1244394,vf6eQrnFfiS9o1kB/gRUJ4iTUixS9tHNKizqQ/1vLDI=,38,10,0,0,0,2016-02-29,2016-05-19,0
20402088,vf6eQrnFfiS9o1kB/gRUJ4iTUixS9tHNKizqQ/1vLDI=,38,10,0,0,0,2016-02-29,2016-05-29,0
12909828,vf6eQrnFfiS9o1kB/gRUJ4iTUixS9tHNKizqQ/1vLDI=,38,10,0,0,0,2016-02-29,2016-06-08,0


,msno,payment_method_id,payment_plan_days,plan_list_price,actual_amount_paid,is_auto_renew,transaction_date,membership_expire_date,is_cancel
21627012,vf6eQrnFfiS9o1kB/gRUJ4iTUixS9tHNKizqQ/1vLDI=,38,10,0,0,0,2016-02-29,2020-03-09,0
21700797,vf6eQrnFfiS9o1kB/gRUJ4iTUixS9tHNKizqQ/1vLDI=,38,10,0,0,0,2016-02-29,2020-03-19,0
21831014,vf6eQrnFfiS9o1kB/gRUJ4iTUixS9tHNKizqQ/1vLDI=,38,10,0,0,0,2016-02-29,2020-03-29,0
21706215,vf6eQrnFfiS9o1kB/gRUJ4iTUixS9tHNKizqQ/1vLDI=,38,10,0,0,0,2016-02-29,2020-04-08,0
21811076,vf6eQrnFfiS9o1kB/gRUJ4iTUixS9tHNKizqQ/1vLDI=,38,10,0,0,0,2016-02-29,2020-04-18,0
21729787,vf6eQrnFfiS9o1kB/gRUJ4iTUixS9tHNKizqQ/1vLDI=,38,10,0,0,0,2016-02-29,2020-04-28,0
21890912,vf6eQrnFfiS9o1kB/gRUJ4iTUixS9tHNKizqQ/1vLDI=,38,10,0,0,0,2016-02-29,2020-05-08,0
21762447,vf6eQrnFfiS9o1kB/gRUJ4iTUixS9tHNKizqQ/1vLDI=,38,10,0,0,0,2016-02-29,2020-05-18,0
21621543,vf6eQrnFfiS9o1kB/gRUJ4iTUixS9tHNKizqQ/1vLDI=,38,10,0,0,0,2016-02-29,2020-05-28,0
21825513,vf6eQrnFfiS9o1kB/gRUJ4iTUixS9tHNKizqQ/1vLDI=,38,10,0,0,0,2016-02-29,2020-06-07,0


membership_expire_date
10.0    155
Name: count, dtype: int64

<div style="background-color: #162B36; color: #E6EDF3; padding: 20px; border-radius: 12px; border-left: 5px solid #38BDF8;">

#### Análisis de múltiples transacciones por cliente y día

Se han identificado clientes con múltiples registros de transacciones en una misma fecha, incluso después de filtrar las operaciones no canceladas (`is_cancel = 0`).

Al analizar uno de los casos extremos, con **156 transacciones registradas el mismo día**, se observa que:

- Todas las operaciones tienen el mismo método de pago, duración del plan (10 días), precio (0) e importe pagado (0).
- La única variable que cambia es `membership_expire_date`.
- Las fechas de vencimiento avanzan sistemáticamente en intervalos de 10 días, desde marzo de 2016 hasta junio de 2020.
- Esto sugiere que los registros podrían representar **extensiones sucesivas de una suscripción**, registradas en una misma fecha, y no compras independientes.

#### Implicaciones para el feature engineering

- **`n_transactions`**: representa el número de registros transaccionales, no necesariamente el número de compras reales.
- **`n_transaction_days`**: número de días distintos en los que el cliente registra actividad. Permite diferenciar entre múltiples operaciones en un mismo día y actividad distribuida en el tiempo.
- **`n_cancellations`**: número de operaciones marcadas como canceladas.
- **`max_transactions_same_day`**: máximo número de operaciones registradas por un cliente en un mismo día.
- **`max_expire_date`**: fecha máxima de vencimiento registrada. Puede reflejar extensiones acumuladas y no necesariamente el estado contractual definitivo.
- **`days_to_max_expiry`**: diferencia entre la fecha máxima de vencimiento y la fecha de corte del modelo.

**Conclusión:** las transacciones deben interpretarse como registros de eventos asociados al ciclo de vida de una suscripción, no necesariamente como compras independientes. Por tanto, se conservarán los registros y se construirán variables agregadas que permitan capturar la actividad real sin sobreestimar la frecuencia de compra.

Además, las fechas de vencimiento posteriores al corte temporal no constituyen necesariamente *data leakage*, siempre que estuvieran registradas y disponibles antes de dicho corte.

</div>

## FEAT ENGINEERING

### Feature Engineering: identificación de clientes con múltiples transacciones por día

Durante el análisis exploratorio se ha identificado que algunos clientes presentan varias transacciones registradas en una misma fecha. Estas operaciones no son necesariamente duplicados exactos, sino que pueden estar relacionadas con cancelaciones, renovaciones, cambios de plan, extensiones de suscripción u otros procesos administrativos de KKBox.

Por tanto, se plantea **diferenciar entre clientes que registran una única operación por día y clientes que presentan múltiples operaciones en una misma fecha**, con el objetivo de evaluar si este comportamiento aporta información relevante para predecir el churn.

#### 1. Variables propuestas

- **`has_multi_transaction_day`**: indicador binario que identifica si el cliente ha registrado alguna vez más de una transacción en un mismo día.
  - `0`: como máximo una transacción por día.
  - `1`: al menos un día con múltiples transacciones.

    
- **`multi_transaction_day_rate`**: proporción de días con múltiples transacciones respecto al total de días con actividad transaccional.

- **`extra_transaction_records`**: número de registros adicionales respecto al escenario de una única transacción por día. Se calcula como `n_transaction_records - n_transaction_days`.

- **`has_multi_non_cancelled_day`**: indicador binario que identifica clientes con al menos dos operaciones no canceladas (`is_cancel = 0`) en una misma fecha.

- **`has_mixed_cancel_day`**: indicador binario que identifica clientes que presentan operaciones canceladas (`is_cancel = 1`) y no canceladas (`is_cancel = 0`) en una misma fecha.

#### 2. Hipótesis

La presencia de múltiples operaciones en un mismo día podría estar asociada a determinados patrones de comportamiento del cliente o a particularidades del sistema de facturación.

Por ejemplo:

- Las operaciones canceladas y no canceladas en una misma fecha podrían reflejar modificaciones o ajustes de suscripción.
- Las múltiples operaciones no canceladas podrían corresponder a extensiones de suscripción o modalidades especiales de contratación.
- La concentración de numerosos registros en un único día podría reflejar procesos administrativos, en lugar de una elevada frecuencia de compra.

Estas interpretaciones son hipótesis que deberán contrastarse con los datos.

#### 3. Estrategia de modelado

Inicialmente, **no se dividirá el dataset ni se entrenarán modelos independientes** para ambos grupos de clientes.

Se incorporarán las variables propuestas al conjunto de features, permitiendo que el modelo identifique posibles relaciones entre los patrones transaccionales y la probabilidad de abandono.

#### 4. Validación de las hipótesis

Una vez integradas las features con `train_v2`, se analizará:

- La distribución de clientes con y sin múltiples transacciones por día.
- La tasa de churn de cada grupo.
- Las diferencias entre clientes con operaciones múltiples no canceladas y aquellos con operaciones mixtas.
- La contribución de estas variables al rendimiento predictivo mediante validación.

**Conclusión:** el objetivo no es eliminar las operaciones múltiples, sino transformar este comportamiento transaccional en variables predictoras que permitan caracterizar mejor a los clientes y evaluar su posible relación con el churn.

### Transactions

In [206]:
# Fecha de corte: queremos predecir el churn de marzo
cutoff = pd.Timestamp("2017-03-01")

# Filtrar por clientes de train_v2 y por fecha
tx = transactions_all.loc[
    transactions_all["msno"].isin(train_v2["msno"]) &
    (transactions_all["transaction_date"] < cutoff)
].copy()

# 2. Variables auxiliares

# Operaciones no canceladas
tx["non_cancelled"] = (tx["is_cancel"] == 0).astype(int)

# Importe asociado a operaciones no canceladas
tx["non_cancelled_amount"] = np.where(
    tx["is_cancel"] == 0,
    tx["actual_amount_paid"],
    0
)

# Indicador de operaciones con precio de catálogo cero
tx["zero_list_price"] = (tx["plan_list_price"] == 0).astype(int)

# Precio normalizado por 30 días.
# Evitamos divisiones por cero cuando payment_plan_days = 0.
tx["payment_per_30d"] = np.where(
    tx["payment_plan_days"] > 0,
    tx["actual_amount_paid"] * 30 /
    tx["payment_plan_days"].replace(0, np.nan),
    np.nan
)

# 3. Agregaciones por cliente
tx_features = tx.groupby("msno").agg(

    # Actividad transaccional
    n_transaction_records =("transaction_date", "size"),
    n_transaction_days=("transaction_date", "nunique"),
    first_transaction=("transaction_date", "min"),
    last_transaction=("transaction_date", "max"),

    # Cancelaciones
    n_cancellations=("is_cancel", "sum"),
    cancel_rate=("is_cancel", "mean"),
    n_non_cancelled_transactions=("non_cancelled", "sum"),
    
    # Importes registrados
    avg_amount_paid=("actual_amount_paid", "mean"),
    total_amount_paid=("actual_amount_paid", "sum"),
    payment_amount_std=("actual_amount_paid", "std"),
    total_non_cancelled_amount=("non_cancelled_amount", "sum"),

    # Precio normalizado
    avg_payment_per_30d=("payment_per_30d", "mean"),
    
    # Planes contratados
    avg_plan_days=("payment_plan_days", "mean"),
    max_plan_days=("payment_plan_days", "max"),
    n_distinct_plan_days=("payment_plan_days", "nunique"),
    n_payment_methods=("payment_method_id", "nunique"),
    n_zero_list_price_transactions=("zero_list_price", "sum"),

    # Renovación automática
    auto_renew_rate=("is_auto_renew", "mean"),

    # Vencimiento máximo registrado
    max_expire_date=("membership_expire_date", "max")
)

# 4. Actividad por cliente y día

daily_activity = (
    tx.groupby(["msno", "transaction_date"])
    .agg(
        n_daily_transactions=("is_cancel", "size"),
        n_daily_cancellations=("is_cancel", "sum"),
        n_daily_non_cancelled=("non_cancelled", "sum")
    )
    .reset_index()
)

# Indicadores diarios
daily_activity["multi_transaction_day"] = (
    daily_activity["n_daily_transactions"] > 1
).astype(int)

daily_activity["multi_non_cancelled_day"] = (
    daily_activity["n_daily_non_cancelled"] >= 2
).astype(int)

daily_activity["multi_mixed_cancel_day"] = (
    (daily_activity["n_daily_cancellations"] >= 1) &
    (daily_activity["n_daily_non_cancelled"] >= 1)
).astype(int)

# Agregación final por cliente
daily_features = daily_activity.groupby("msno").agg(
    max_transactions_same_day=("n_daily_transactions", "max"),
    avg_transactions_per_day=("n_daily_transactions", "mean"),
    n_multi_transaction_days=("multi_transaction_day", "sum"),
    has_multi_non_cancelled_day=("multi_non_cancelled_day", "max"),
    has_mixed_cancel_day=("multi_mixed_cancel_day", "max")
)


# 5. Variables de la última fecha observada

# Identificamos todas las operaciones del último día de cada cliente
last_dates = tx.groupby("msno")["transaction_date"].transform("max")

last_day_tx = tx.loc[
    tx["transaction_date"].eq(last_dates)
]

last_day_features = last_day_tx.groupby("msno").agg(
    last_day_n_transactions=("transaction_date", "size"),
    last_day_any_cancel=("is_cancel", "max"),
    last_day_any_auto_renew=("is_auto_renew", "max"),
    last_day_all_auto_renew=("is_auto_renew", "min"),
    last_day_max_expire=("membership_expire_date", "max"),
    last_day_max_payment=("actual_amount_paid", "max")
)

# 6. Integración de todas las features
client_txs = (
    tx_features
    .join(daily_features, how="left")
    .join(last_day_features, how="left")
)

# 7. Variables temporales

# Días desde la última transacción
client_txs["recency_days"] = (
    cutoff - client_txs["last_transaction"]
).dt.days

# Periodo observado entre primera y última transacción
client_txs["transaction_span_days"] = (
    client_txs["last_transaction"] -
    client_txs["first_transaction"]
).dt.days

# Días hasta el vencimiento máximo registrado
client_txs["days_to_max_expiry"] = (
    client_txs["max_expire_date"] - cutoff
).dt.days

# Días hasta el vencimiento máximo del último día observado
client_txs["days_to_expiry"] = (
    client_txs["last_day_max_expire"] - cutoff
).dt.days


# Identifica si el cliente ha tenido alguna vez operaciones múltiples en una fecha
client_txs["has_multi_transaction_day"] = (
    client_txs["max_transactions_same_day"] > 1
).astype(int)

# Proporción de días de actividad con múltiples transacciones
client_txs["multi_transaction_day_rate"] = (
    client_txs["n_multi_transaction_days"] /
    client_txs["n_transaction_days"]
)

# Cuantifica número de registros adicionales respecto a una operación por día
client_txs["extra_transaction_records"] = (
    client_txs["n_transaction_records"] -
    client_txs["n_transaction_days"]
)

# 8. Resultado final
client_txs = client_txs.reset_index()

# Comprobación: una única fila por cliente
assert client_txs["msno"].is_unique
# Comprobación: no se incorporan transacciones de marzo
assert tx["transaction_date"].max() < cutoff


print(f"Tamaño del dataset: {client_txs.shape}")

display(client_txs.head())

Tamaño del dataset: (968436, 38)


,msno,n_transaction_records,n_transaction_days,first_transaction,last_transaction,n_cancellations,cancel_rate,n_non_cancelled_transactions,avg_amount_paid,total_amount_paid,...,last_day_all_auto_renew,last_day_max_expire,last_day_max_payment,recency_days,transaction_span_days,days_to_max_expiry,days_to_expiry,has_multi_transaction_day,multi_transaction_day_rate,extra_transaction_records
0,+++hVY1rZox/33YtvDgmKA2Frg/2qhkz12B9ylCvh8o=,4,4,2016-11-16,2017-02-15,0,0.0,4,99.0,396,...,1,2017-03-15,99,14,91,14,14,0,0.0,0
1,+++l/EXNMLTijfLBa8p2TUVVVp2aFGSuUI/h7mLmthw=,20,20,2015-01-31,2017-02-28,0,0.0,20,149.0,2980,...,1,2017-04-19,149,1,759,49,49,0,0.0,0
2,+++snpr7pmobhLKUgSHTv/mpkqgBT0tQJ0zQj6qKrqc=,26,26,2015-01-26,2017-02-26,0,0.0,26,149.0,3874,...,1,2017-03-26,149,3,762,25,25,0,0.0,0
3,++/9R3sX37CjxbY/AaGvbwr3QkwElKBCtSvVzhCBDOk=,12,12,2016-03-15,2017-02-15,0,0.0,12,149.0,1788,...,1,2017-03-15,149,14,337,14,14,0,0.0,0
4,++/UDNo9DLrxT8QVGiDi1OnWfczAdEwThaVyD0fXO50=,20,20,2015-01-31,2017-02-28,0,0.0,20,149.0,2980,...,1,2017-04-23,149,1,759,53,53,0,0.0,0


Algunas consideraciones:
- payment_amount_std: será NaN para clientes con una única transacción. Es normal; todavía no imputaría.
- days_to_expiry: puede ser negativo si la suscripción ya había vencido en la fecha de corte.
- n_plan_changes: solo detecta cambios en la perioricidad del pago, no todos los posibles cambios de tarifa.
- n_cancellations: cuenta operaciones marcadas como cancelación, no abandonos definitivos.

### Members

In [ ]:
cutoff = pd.Timestamp("2017-03-01")

members["registration_init_time"] = pd.to_datetime(
    members["registration_init_time"].astype("Int64").astype(str),
    format="%Y%m%d",
    errors="raise"
)

# Caluclamos dias de registro desde la fecha de corte y eliminamos clientes con registros posteriores
members["membership_age_days"] = (
    cutoff -  members["registration_init_time"]
).dt.days

# Eliminar únicamente clientes registrados después del cutoff
members = members.loc[
    members["membership_age_days"].isna() |
    (members["membership_age_days"] >= 0)
].copy()

## DATA INTEGRATION

In [215]:
# Unimos train_v2 con members (relación 1:1)
data = train_v2.merge(
    members,
    on="msno",
    how="left",
    validate="one_to_one",   # falla si hay duplicados en alguna tabla
    indicator=False           # añade la columna _merge: both / left_only
)

# Unimos transacciones por cliente
df = data.merge(
    client_txs,
    on="msno",
    how="left",
    validate="one_to_one"
)

print(f"Clientes en train_v2: {len(train_v2):,}")
print(f"Filas del dataset final: {len(df):,}")

assert len(df) == len(train_v2)

Clientes en train_v2: 970,960
Filas del dataset final: 970,960


In [216]:
print(df.shape)
display(df.head(4))

(970960, 45)


,msno,is_churn,city,bd,gender,registered_via,registration_init_time,membership_age_days,n_transaction_records,n_transaction_days,...,last_day_all_auto_renew,last_day_max_expire,last_day_max_payment,recency_days,transaction_span_days,days_to_max_expiry,days_to_expiry,has_multi_transaction_day,multi_transaction_day_rate,extra_transaction_records
0,ugx0CjOMzazClkFzU2xasmDZaoIqOUAZPsH1q0teWCg=,1,5.0,28.0,male,3.0,2013-12-23,1164.0,6.0,6.0,...,0.0,2017-03-30,149.0,1.0,700.0,29.0,29.0,0.0,0.000000,0.0
1,f/NmvEzHfhINFEYZTR05prUdr+E+3+oewvweYz9cCQE=,1,13.0,20.0,male,3.0,2013-12-23,1164.0,5.0,5.0,...,0.0,2017-03-12,180.0,19.0,344.0,11.0,11.0,0.0,0.000000,0.0
2,zLo9f73nGGT1p21ltZC3ChiRnAVvgibMyazbCxvWPcg=,1,13.0,18.0,male,3.0,2013-12-27,1160.0,18.0,18.0,...,0.0,2017-03-07,149.0,30.0,530.0,6.0,6.0,0.0,0.000000,0.0
3,8iF/+8HY8lJKFrTc7iR9ZYGCG2Ecrogbc2Vy5YhsfhQ=,1,1.0,0.0,NaN,7.0,2014-01-09,1147.0,28.0,15.0,...,0.0,2016-03-25,149.0,370.0,413.0,313.0,-341.0,1.0,0.866667,13.0


## EXPORT FINAL DATASETS

### Extraer y guardar datasets de validation y train

In [210]:
# Update validation-train split size
val = df.sample(frac=0.3)
val.shape

(291288, 45)

In [217]:
# Actualizar el tamaño validation-train
val= df.sample(frac = 0.3, random_state=42)

validation_file_name = 'validation.csv'
full_path = path / "02_data" / "validation" / validation_file_name

val.to_csv(full_path, index=False)

train = df.loc[~df.index.isin(val.index)]
train_file_name = 'train.csv'
full_path = path / '02_data' /'train' / train_file_name

train.to_csv(full_path, index= False)

### Extraer y guardar una muestra (opcional)

In [212]:
# Update sample size
sample = train.sample(n=20000)

sample_file_name = 'sample.csv'
full_path = path / '02_data' / 'train' / sample_file_name

sample.to_csv(full_path)